In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if not candidate.suffix and (candidate.is_absolute() or ':' in text):
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import random
from scipy import stats
import warnings
import seaborn as sns

In [ ]:
import xgboost as xgb
from sklearn import mixture
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn import svm, linear_model, datasets
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, accuracy_score, 
                             roc_auc_score, RocCurveDisplay, classification_report)
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc

In [ ]:
# load files
with open(ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl', 'rb') as file:
    pos = pickle.load(file)

with open(ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl', 'rb') as file:
    neg = pickle.load(file)

In [ ]:
pos[0].head()

In [ ]:
# trim files and convert to numpy array
for i in range(len(pos)):
    df = pos[i].dropna()
    df['Y'] = [1] * len(df)
    pos[i] = df

for i in range(len(neg)):
    df = neg[i].dropna()
    df['Y'] = [0] * len(df)
    neg[i] = df

In [ ]:
def partition_data(pos, neg, test_num, features_left, random_state = 0, return_held_out = False):
    # get list of complex names
    complexes = ['Cyclin–CDK complexes', 'RB–E2F repressor complex', 'ORC–MCM pre-replication complex', 
                           'Anaphase-Promoting Complex (APC/C)', 'Cohesin complex', 'BRCA1–BARD1 complex',
                           'Fanconi Anemia (FA) core complex', 'ATR–CHK1 checkpoint complex', 'PI3K–AKT–mTOR pathway complexes',
                           'MAPK cascade', 'LKB1–STRAD–MO25 complex', 'AMPK complex', 'BCL-2 family complexes',
                           'Death-Inducing Signaling Complex (DISC)', 'Apoptosome', 'NF-κB transcriptional complex',
                           'Inflammasome', 'Autophagy initiation complex', 'SWI/SNF (BAF/PBAF) complex', 'TIP60/NuA4 complex',
                           'PRC2 complex', 'Mediator/CDK8 module', 'p53 regulatory network', 'Proteasome complex',
                           'HSP90–CDC37 chaperone complex', 'ISR/EIF2α–ATF4 complex']
    
    # partition training and testing data
    train_set = []
    test_set = []
    held_out_set = []
    rng = random.Random(random_state)
    for i in range(len(pos)):
        test_complexes = rng.sample(complexes, test_num)
        held_out_set.append(test_complexes)
        
        pos_set = pos[i]
        neg_set = neg[i].sample(frac=1, random_state=0)
        
        train_pos = (pos_set[~pos_set['Complex'].isin(test_complexes)]
            .drop(columns = ['Complex', 'pair'])
            .to_numpy())
        test_pos = (pos_set[pos_set['Complex'].isin(test_complexes)]
            .drop(columns = ['Complex', 'pair'])
            .to_numpy())
        train_neg = (neg_set[:len(train_pos)]
            .drop(columns = ['pair'])
            .to_numpy())
        test_neg = (neg_set[len(train_pos):len(train_pos) + len(test_pos)]
            .drop(columns = ['pair'])
            .to_numpy())
    
        # KS - Test
        p_val = []
        for j in range(train_pos.shape[1] - 1):    # -1 is because last column identifies positive/negative
            _, p_value = stats.ks_2samp(train_pos[:, j], train_neg[:, j])
            p_val.append(p_value)
    
        drop_ind = np.argpartition(p_val, features_left-1)[features_left:]
        train_set.append(np.delete(np.concatenate([train_pos, train_neg]), drop_ind, axis=1))
        test_set.append(np.delete(np.concatenate([test_pos, test_neg]), drop_ind, axis=1))

    if return_held_out:
        return train_set, test_set, held_out_set
    return train_set, test_set

In [ ]:
def ML(train_set, test_set, index):
    X_train = train_set[index][:,:-1]
    y_train = train_set[index][:,-1]
    
    X_test = test_set[index][:,:-1]
    y_test = test_set[index][:,-1]
    
    X = np.concatenate([X_train, X_test])
    y = np.concatenate([y_train, y_test])
    
    models = {
        # 'Logistic Regression': LogisticRegression(),
        # 'Ridge Classifier': RidgeClassifier(),
        # 'Support Vector Classifier': svm.SVC(random_state=0),
        # 'Decision Tree Classifier': DecisionTreeClassifier(),
        # 'Random Forest Classifier': RandomForestClassifier(),
        # 'Gradient Boosting Classifier': GradientBoostingClassifier(),
        # 'K-Neighbors Classifier': KNeighborsClassifier(),
        # 'Gaussian Naive Bayes': GaussianNB(),
        # 'MLP Classifier': MLPClassifier(max_iter=1000),
        'XGBoost': XGBClassifier()
    }
    
    ROC_AUC_Scores = {}
    training_accuracies = {}
    testing_accuracies = {}
    crossValidationScores = {}
    fprList = {}
    tprList = {}
    
    # plt.figure(figsize=(9, 6))
    
    # Loop through the models
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # ROC AUC score requires probability estimates for binary classifiers
        if hasattr(model, "predict_proba"):
            y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability estimates for the positive class
        else:  # Some models like SVC may not have predict_proba; we can use decision_function instead
            y_pred_proba = model.decision_function(X_test)
        
        # Model assessment
        train_acc = model.score(X_train, y_train)
        test_acc = model.score(X_test, y_test)
        AUC = roc_auc_score(y_test, y_pred_proba)
        report = classification_report(y_test, y_pred)
        crossVal = cross_val_score(model, X, y)
        
        # Compute ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        
        # Append to the dictionaries
        ROC_AUC_Scores[name] = AUC
        
        training_accuracies[name] = train_acc
        testing_accuracies[name] = test_acc
        crossValidationScores[name] = crossVal
        fprList[name] = fpr
        tprList[name] = tpr
        
        # # Plot ROC curve
        # plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {AUC:.2f})')
        
        # print(f"Model: {name}")
        # print(f"Training Accuracy: {train_acc}")
        # print(f'Testing Accuracy: {test_acc}')
        # print(f'The ROC_AUC score is {AUC}')
        # print(f'The cross validation score is {crossVal}')
        # print("Classification Report:")
        # print(report)
        # print("-" * 30)
        
    # plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    
    # # Customize the plot
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.title('ROC Curves for Multiple Models')
    # plt.legend(loc='lower right')
    # plt.show()
    return ROC_AUC_Scores

In [ ]:
warnings.filterwarnings('ignore')
df = pd.DataFrame(columns=['Logistic Regression', 'Ridge Classifier', 'Support Vector Classifier', 
                           'Decision Tree Classifier', 'Random Forest Classifier', 'Gradient Boosting Classifier', 
                           'K-Neighbors Classifier', 'Gaussian Naive Bayes', 'MLP Classifier', 'XGBoost'])
features_left = [20, 50, 75, 100, 200, 400, 800]
for n in features_left:
    train_set, test_set = partition_data(pos, neg, 5, n)
    temp = np.zeros(10)
    for i in range(10):
        d = ML(train_set, test_set, i)
        temp += np.array(list(d.values()), dtype=float)
    df.loc[len(df)] = temp
df = pd.concat([pd.DataFrame({'Features Left':features_left}), df, 
                pd.DataFrame({'Mean':[df.iloc[i, 1:].sum()/10 for i in range(len(features_left))]})], axis=1)
df

In [ ]:
warnings.filterwarnings('ignore')
df = pd.DataFrame(columns=['Logistic Regression', 'Ridge Classifier', 'Support Vector Classifier', 
                           'Decision Tree Classifier', 'Random Forest Classifier', 'Gradient Boosting Classifier', 
                           'K-Neighbors Classifier', 'Gaussian Naive Bayes', 'MLP Classifier', 'XGBoost'])
train_set, test_set = partition_data(pos, neg, 5, 50)
for i in range(10):
    d = ML(train_set, test_set, i)
    df.loc[len(df)] = np.array(list(d.values()), dtype=float)
df = pd.concat([df, pd.DataFrame({'Mean':[df.iloc[i, :].sum()/10 for i in range(10)]})], axis=1)
df

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot([df[col] for col in df.columns], labels=df.columns)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.title("ROC-AUC Scores")
plt.show()

In [ ]:
def column_stats(df, confidence=0.95):
    results = []

    df_numeric = df.select_dtypes(include='number')

    for col in df_numeric.columns:
        data = df_numeric[col].dropna()
        n = len(data)
        mean = np.mean(data)
        std = np.std(data, ddof=1)  # sample std
        sem = std / np.sqrt(n)

        t_val = stats.t.ppf((1 + confidence) / 2, df=n-1)
        ci_low = mean - t_val * sem
        ci_high = mean + t_val * sem

        results.append({
            "Model": col,
            "Mean": mean,
            "STD": std,
            "CI_lower": ci_low,
            "CI_upper": ci_high
        })

    return pd.DataFrame(results)

column_stats(df)

In [ ]:
df = pd.DataFrame(columns=['Cyclin–CDK complexes', 'RB–E2F repressor complex', 'ORC–MCM pre-replication complex', 
                           'Anaphase-Promoting Complex (APC/C)', 'Cohesin complex', 'BRCA1–BARD1 complex',
                           'Fanconi Anemia (FA) core complex', 'ATR–CHK1 checkpoint complex', 'PI3K–AKT–mTOR pathway complexes',
                           'MAPK cascade', 'LKB1–STRAD–MO25 complex', 'AMPK complex', 'BCL-2 family complexes',
                           'Death-Inducing Signaling Complex (DISC)', 'Apoptosome', 'NF-κB transcriptional complex',
                           'Inflammasome', 'Autophagy initiation complex', 'SWI/SNF (BAF/PBAF) complex', 'TIP60/NuA4 complex',
                           'PRC2 complex', 'Mediator/CDK8 module', 'p53 regulatory network', 'Proteasome complex',
                           'HSP90–CDC37 chaperone complex', 'ISR/EIF2α–ATF4 complex'])
df.loc['sum'] = [0] * 26
df.loc['count'] = [0] * 26

for i in range(100):
    train_set, test_set, held_out = partition_data(pos, neg, 1, 50, random_state = i, return_held_out = True)
    for j in range(10):
        val = ML(train_set, test_set, j)['XGBoost']
        df.loc['sum', held_out[j]] += val
        df.loc['count', held_out[j]] += 1
        
df.loc['mean'] = df.loc['sum'] / df.loc['count']
df = df.sort_values(by='mean', axis=1)
df